In [1]:
############# MEAN 0.8438 ###############


import os
import gc
import random
import warnings
import numpy as np
import pandas as pd
import cv2
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torch.cuda.amp import autocast
from PIL import Image
from tqdm.auto import tqdm
from pathlib import Path
from dataclasses import dataclass

from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.cross_decomposition import PLSRegression
from sklearn.mixture import GaussianMixture
from sklearn.ensemble import GradientBoostingRegressor, HistGradientBoostingRegressor
from lightgbm import LGBMRegressor
from catboost import CatBoostRegressor
from transformers import AutoModel, AutoImageProcessor, AutoTokenizer

import timm

warnings.filterwarnings('ignore')
os.environ["TOKENIZERS_PARALLELISM"] = "false"

@dataclass
class Config:
    DATA_PATH: Path = Path("/kaggle/input/csiro-biomass")
    SPLIT_PATH: Path = Path("/kaggle/input/csiro-datasplit/csiro_data_split.csv")
    MODELS_DIR: Path = Path("/kaggle/input/dino-retrain-hu-2/models_trained")
    SIGLIP_PATH: str = "/kaggle/input/google-siglip-so400m-patch14-384/transformers/default/1"
    MODEL_NAME: str = "vit_huge_plus_patch16_dinov3.lvd1689m"
    SEED: int = 42
    DEVICE: str = "cuda" if torch.cuda.is_available() else "cpu"
    IMG_HEIGHT: int = 768
    IMG_WIDTH: int = 384
    BATCH_SIZE: int = 4
    N_FOLDS: int = 3
    DROPOUT: float = 0.2
    TARGETS = ["Dry_Green_g", "Dry_Dead_g", "Dry_Clover_g", "GDM_g", "Dry_Total_g"]
    TARGET_NAMES = ['Dry_Clover_g', 'Dry_Dead_g', 'Dry_Green_g', 'Dry_Total_g', 'GDM_g']
    TARGET_MAX = {
        "Dry_Clover_g": 71.7865,
        "Dry_Dead_g": 83.8407,
        "Dry_Green_g": 157.9836,
        "Dry_Total_g": 185.70,
        "GDM_g": 157.9836,
    }
    W_DINO: float = 0.75
    W_SIGLIP: float = 0.25

cfg = Config()

def seed_everything(seed=cfg.SEED):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

seed_everything()

print(f"Device: {cfg.DEVICE}")
print(f"Model: {cfg.MODEL_NAME}")
print(f"Models Dir: {cfg.MODELS_DIR}")

class LocalMambaBlock(nn.Module):
    def __init__(self, dim: int, kernel_size: int = 5, dropout: float = 0.1):
        super().__init__()
        self.norm = nn.LayerNorm(dim)
        self.dwconv = nn.Conv1d(dim, dim, kernel_size=kernel_size, padding=kernel_size // 2, groups=dim)
        self.gate = nn.Linear(dim, dim)
        self.proj = nn.Linear(dim, dim)
        self.drop = nn.Dropout(dropout)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        shortcut = x
        x = self.norm(x)
        g = torch.sigmoid(self.gate(x))
        x = x * g
        x = x.transpose(1, 2)
        x = self.dwconv(x)
        x = x.transpose(1, 2)
        x = self.proj(x)
        x = self.drop(x)
        return shortcut + x

class BiomassModel(nn.Module):
    def __init__(self, model_name: str, pretrained: bool = False):
        super().__init__()
        self.backbone = timm.create_model(
            model_name, pretrained=pretrained, num_classes=0, global_pool=''
        )
        nf = self.backbone.num_features
        
        self.fusion = nn.Sequential(
            LocalMambaBlock(nf, kernel_size=5, dropout=cfg.DROPOUT),
            LocalMambaBlock(nf, kernel_size=5, dropout=cfg.DROPOUT)
        )
        self.pool = nn.AdaptiveAvgPool1d(1)
        
        self.head_green = nn.Sequential(
            nn.Linear(nf, nf // 2), nn.GELU(), nn.Dropout(cfg.DROPOUT),
            nn.Linear(nf // 2, 1), nn.Softplus()
        )
        self.head_dead = nn.Sequential(
            nn.Linear(nf, nf // 2), nn.GELU(), nn.Dropout(cfg.DROPOUT),
            nn.Linear(nf // 2, 1), nn.Softplus()
        )
        self.head_clover = nn.Sequential(
            nn.Linear(nf, nf // 2), nn.GELU(), nn.Dropout(cfg.DROPOUT),
            nn.Linear(nf // 2, 1), nn.Softplus()
        )

    def forward(self, left, right):
        x_l = self.backbone(left)
        x_r = self.backbone(right)
        x_cat = torch.cat([x_l, x_r], dim=1)
        x_fused = self.fusion(x_cat)
        x_pool = self.pool(x_fused.transpose(1, 2)).flatten(1)
        
        green = self.head_green(x_pool)
        dead = self.head_dead(x_pool)
        clover = self.head_clover(x_pool)
        gdm = green + clover
        total = gdm + dead
        
        return torch.cat([green, dead, clover, gdm, total], dim=1)

class TestDataset(Dataset):
    def __init__(self, df, image_root, img_height=768, img_width=384):
        self.df = df.reset_index(drop=True)
        self.image_root = image_root
        self.img_height = img_height
        self.img_width = img_width
        
        self.mean = np.array([0.4417, 0.5036, 0.3057])
        self.std = np.array([0.2364, 0.2355, 0.2219])

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img_path = self.image_root / row["image_path"]
        
        img = cv2.imread(str(img_path))
        if img is None:
            img = np.zeros((1000, 2000, 3), dtype=np.uint8)
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        
        h, w, _ = img.shape
        mid = w // 2
        left = img[:, :mid]
        right = img[:, mid:]
        
        left = cv2.resize(left, (self.img_width, self.img_height))
        right = cv2.resize(right, (self.img_width, self.img_height))
        
        left = left.astype(np.float32) / 255.0
        right = right.astype(np.float32) / 255.0
        left = (left - self.mean) / self.std
        right = (right - self.mean) / self.std
        
        left = torch.from_numpy(left.transpose(2, 0, 1)).float()
        right = torch.from_numpy(right.transpose(2, 0, 1)).float()
        
        return left, right, row.to_dict()

def collate_fn(batch):
    lefts = torch.stack([b[0] for b in batch])
    rights = torch.stack([b[1] for b in batch])
    infos = [b[2] for b in batch]
    return lefts, rights, infos

@torch.no_grad()
def predict_dino(model, loader, device):
    model.eval()
    preds_all = []
    
    for lefts, rights, _ in tqdm(loader, desc="DINO Inference"):
        lefts = lefts.to(device)
        rights = rights.to(device)
        
        with autocast():
            pred = model(lefts, rights)
        
        preds_all.append(pred.cpu().numpy())
    
    return np.vstack(preds_all)

def split_image(image, patch_size=520, overlap=16):
    h, w, c = image.shape
    stride = patch_size - overlap
    patches = []
    for y in range(0, h, stride):
        for x in range(0, w, stride):
            y2 = min(y + patch_size, h)
            x2 = min(x + patch_size, w)
            y1 = max(0, y2 - patch_size)
            x1 = max(0, x2 - patch_size)
            patches.append(image[y1:y2, x1:x2, :])
    return patches

def compute_siglip_embeddings(model_path, df, img_dir):
    print(f"Computing SigLIP embeddings for {len(df)} images...")
    model = AutoModel.from_pretrained(model_path, local_files_only=True).eval().to(cfg.DEVICE)
    processor = AutoImageProcessor.from_pretrained(model_path)
    
    embeddings = []
    for _, row in tqdm(df.iterrows(), total=len(df)):
        try:
            img_path = img_dir / row['image_path']
            img = cv2.imread(str(img_path))
            img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
            patches = split_image(img)
            images = [Image.fromarray(p) for p in patches]
            
            inputs = processor(images=images, return_tensors="pt").to(cfg.DEVICE)
            with torch.no_grad():
                features = model.get_image_features(**inputs)
            
            embeddings.append(features.mean(dim=0).cpu().numpy())
        except Exception as e:
            print(f"Error: {e}")
            embeddings.append(np.zeros(1152))
    
    del model
    torch.cuda.empty_cache()
    return np.stack(embeddings)

def generate_semantic_features(embeddings, model_path):
    model = AutoModel.from_pretrained(model_path).to(cfg.DEVICE)
    tokenizer = AutoTokenizer.from_pretrained(model_path)
    
    concepts = {
        "bare": ["bare soil", "dirt ground", "sparse vegetation", "exposed earth"],
        "sparse": ["low density pasture", "thin grass", "short clipped grass"],
        "medium": ["average pasture cover", "medium height grass", "grazed pasture"],
        "dense": ["dense tall pasture", "thick grassy volume", "high biomass"],
        "green": ["lush green vibrant pasture", "photosynthesizing leaves", "fresh growth"],
        "dead": ["dry brown dead grass", "yellow straw", "senesced material"],
        "clover": ["white clover", "trifolium repens", "broadleaf legume"],
        "grass": ["ryegrass", "blade-like leaves", "fescue", "grassy sward"]
    }
    
    concept_vectors = {}
    with torch.no_grad():
        for name, prompts in concepts.items():
            inputs = tokenizer(prompts, padding="max_length", return_tensors="pt").to(cfg.DEVICE)
            emb = model.get_text_features(**inputs)
            emb = emb / emb.norm(p=2, dim=-1, keepdim=True)
            concept_vectors[name] = emb.mean(dim=0, keepdim=True)
    
    img_tensor = torch.tensor(embeddings, dtype=torch.float32).to(cfg.DEVICE)
    img_tensor = img_tensor / img_tensor.norm(p=2, dim=-1, keepdim=True)
    
    scores = {}
    for name, vec in concept_vectors.items():
        scores[name] = torch.matmul(img_tensor, vec.T).cpu().numpy().flatten()
    
    df_scores = pd.DataFrame(scores)
    df_scores['ratio_greenness'] = df_scores['green'] / (df_scores['green'] + df_scores['dead'] + 1e-6)
    df_scores['ratio_clover'] = df_scores['clover'] / (df_scores['clover'] + df_scores['grass'] + 1e-6)
    
    del model
    torch.cuda.empty_cache()
    return df_scores.values

class SupervisedEmbeddingEngine:
    def __init__(self):
        self.scaler = StandardScaler()
        self.pca = PCA(n_components=0.80, random_state=42)
        self.pls = PLSRegression(n_components=8, scale=False)
        self.gmm = GaussianMixture(n_components=6, covariance_type='diag', random_state=42)
        self.pls_fitted_ = False

    def fit(self, X, y=None, X_semantic=None):
        X_scaled = self.scaler.fit_transform(X)
        self.pca.fit(X_scaled)
        self.gmm.fit(X_scaled)
        if y is not None:
            self.pls.fit(X_scaled, y)
            self.pls_fitted_ = True
        return self

    def transform(self, X, X_semantic=None):
        X_scaled = self.scaler.transform(X)
        feats = [self.pca.transform(X_scaled)]
        if self.pls_fitted_:
            feats.append(self.pls.transform(X_scaled))
        feats.append(self.gmm.predict_proba(X_scaled))
        if X_semantic is not None:
            sem_norm = (X_semantic - np.mean(X_semantic, axis=0)) / (np.std(X_semantic, axis=0) + 1e-6)
            feats.append(sem_norm)
        return np.hstack(feats)

def train_gbdt_cv(model_cls, params, train_data, test_data, sem_tr, sem_te, emb_cols):
    target_max_arr = np.array([cfg.TARGET_MAX[t] for t in cfg.TARGET_NAMES])
    y_pred_test = np.zeros([len(test_data), len(cfg.TARGET_NAMES)])
    n_splits = int(train_data['fold'].nunique())
    
    X_train = train_data[emb_cols].values.astype(np.float32)
    X_test = test_data[emb_cols].values.astype(np.float32)
    y_train = train_data[cfg.TARGET_NAMES].values.astype(np.float32)
    
    for fold in range(n_splits):
        train_mask = train_data['fold'] != fold
        X_tr = X_train[train_mask]
        y_tr = y_train[train_mask] / target_max_arr
        sem_tr_fold = sem_tr[train_mask]
        
        eng = SupervisedEmbeddingEngine()
        eng.fit(X_tr, y=y_tr, X_semantic=sem_tr_fold)
        
        x_tr_eng = eng.transform(X_tr, X_semantic=sem_tr_fold)
        x_te_eng = eng.transform(X_test, X_semantic=sem_te)
        
        for k, target in enumerate(cfg.TARGET_NAMES):
            if target == 'Dry_Clover_g':
                continue
            model = model_cls(**params)
            model.fit(x_tr_eng, y_tr[:, k])
            y_pred_test[:, k] += model.predict(x_te_eng) * target_max_arr[k]
    
    return y_pred_test / n_splits

print("="*60)
print("CSIRO BIOMASS INFERENCE - VIT_HUGE_PLUS + SIGLIP")
print("="*60)

print("\n[1/6] Loading test data...")
test_df_raw = pd.read_csv(cfg.DATA_PATH / 'test.csv')
test_wide = test_df_raw[["image_path"]].drop_duplicates().reset_index(drop=True)
print(f"Test images: {len(test_wide)}")

print("\n[2/6] Running DINO HUGE inference...")

test_dataset = TestDataset(test_wide, cfg.DATA_PATH, cfg.IMG_HEIGHT, cfg.IMG_WIDTH)
test_loader = DataLoader(
    test_dataset, 
    batch_size=cfg.BATCH_SIZE, 
    shuffle=False, 
    num_workers=0,
    collate_fn=collate_fn
)

all_fold_preds = []

for fold in range(cfg.N_FOLDS):
    model_path = cfg.MODELS_DIR / f"fold{fold}_best.pth"
    if not model_path.exists():
        print(f"  Fold {fold} not found, skipping...")
        continue
    
    print(f"  Loading fold {fold}...")
    model = BiomassModel(cfg.MODEL_NAME, pretrained=False).to(cfg.DEVICE)
    state_dict = torch.load(model_path, map_location=cfg.DEVICE)
    
    if list(state_dict.keys())[0].startswith('module.'):
        state_dict = {k.replace('module.', ''): v for k, v in state_dict.items()}
    
    model.load_state_dict(state_dict)
    
    fold_preds = predict_dino(model, test_loader, cfg.DEVICE)
    all_fold_preds.append(fold_preds)
    
    del model, state_dict
    gc.collect()
    torch.cuda.empty_cache()

dino_preds = np.mean(all_fold_preds, axis=0)
print(f"DINO predictions shape: {dino_preds.shape}")

dino_df = test_wide.copy()
dino_df['Dry_Green_g'] = dino_preds[:, 0]
dino_df['Dry_Dead_g'] = dino_preds[:, 1]
dino_df['Dry_Clover_g'] = dino_preds[:, 2] * 0.8

for i in range(len(dino_df)):
    if dino_df.loc[i, 'Dry_Dead_g'] > 20:
        dino_df.loc[i, 'Dry_Dead_g'] *= 1.1
    elif dino_df.loc[i, 'Dry_Dead_g'] < 10:
        dino_df.loc[i, 'Dry_Dead_g'] *= 0.9

dino_df['GDM_g'] = dino_df['Dry_Green_g'] + dino_df['Dry_Clover_g']
dino_df['Dry_Total_g'] = dino_df['GDM_g'] + dino_df['Dry_Dead_g']

print("\n[3/6] Running SigLIP inference...")

train_split = pd.read_csv(cfg.SPLIT_PATH)
cols_keep = [c for c in train_split.columns if not c.startswith('emb')]
train_split = train_split[cols_keep]

if not str(train_split['image_path'].iloc[0]).startswith('/'):
    train_split['image_path'] = train_split['image_path'].apply(
        lambda p: str(cfg.DATA_PATH / 'train' / os.path.basename(p))
    )

test_siglip = test_wide.copy()
test_siglip['image_path'] = test_siglip['image_path'].apply(lambda p: str(cfg.DATA_PATH / p))

print("  Computing train embeddings...")
train_emb = compute_siglip_embeddings(cfg.SIGLIP_PATH, train_split, cfg.DATA_PATH)
print("  Computing test embeddings...")
test_emb = compute_siglip_embeddings(cfg.SIGLIP_PATH, test_siglip, cfg.DATA_PATH)

emb_cols = [f"emb{i}" for i in range(train_emb.shape[1])]
train_feat = pd.concat([train_split, pd.DataFrame(train_emb, columns=emb_cols)], axis=1)
test_feat = pd.concat([test_siglip.reset_index(drop=True), pd.DataFrame(test_emb, columns=emb_cols)], axis=1)

print("  Generating semantic features...")
all_emb = np.vstack([train_emb, test_emb])
all_sem = generate_semantic_features(all_emb, cfg.SIGLIP_PATH)
sem_train = all_sem[:len(train_split)]
sem_test = all_sem[len(train_split):]

print("\n[4/6] Training GBDT models...")
params_hist = {'max_iter': 300, 'learning_rate': 0.05, 'max_depth': 5, 'random_state': 42}
params_gb = {'n_estimators': 1354, 'learning_rate': 0.01, 'max_depth': 3, 'random_state': 42}
params_cat = {'iterations': 1900, 'learning_rate': 0.045, 'depth': 4, 'verbose': 0, 'random_state': 42, 'allow_writing_files': False}
params_lgbm = {'n_estimators': 807, 'learning_rate': 0.014, 'num_leaves': 48, 'verbose': -1, 'random_state': 42}

print("  HistGB...")
pred_hist = train_gbdt_cv(HistGradientBoostingRegressor, params_hist, train_feat, test_feat, sem_train, sem_test, emb_cols)
print("  GB...")
pred_gb = train_gbdt_cv(GradientBoostingRegressor, params_gb, train_feat, test_feat, sem_train, sem_test, emb_cols)
print("  CatBoost...")
pred_cat = train_gbdt_cv(CatBoostRegressor, params_cat, train_feat, test_feat, sem_train, sem_test, emb_cols)
print("  LightGBM...")
pred_lgbm = train_gbdt_cv(LGBMRegressor, params_lgbm, train_feat, test_feat, sem_train, sem_test, emb_cols)

siglip_pred = (pred_hist + pred_gb + pred_cat + pred_lgbm) / 4.0

siglip_df = test_siglip.copy()
siglip_df[cfg.TARGET_NAMES] = siglip_pred
siglip_df['Dry_Clover_g'] = 0.0
siglip_df['GDM_g'] = siglip_df['Dry_Green_g']
siglip_df['Dry_Total_g'] = siglip_df['GDM_g'] + siglip_df['Dry_Dead_g']

print("\n[5/6] Creating ensemble...")
print(f"Weights: DINO={cfg.W_DINO}, SigLIP={cfg.W_SIGLIP}")

ALL_TARGETS = ['Dry_Green_g', 'Dry_Clover_g', 'Dry_Dead_g', 'GDM_g', 'Dry_Total_g']

final_df = test_wide.copy()

for target in ALL_TARGETS:
    if target == 'Dry_Clover_g':
        final_df[target] = dino_df[target]
    else:
        final_df[target] = dino_df[target] * cfg.W_DINO + siglip_df[target] * cfg.W_SIGLIP

final_df['Dry_Clover_g'] = final_df['Dry_Clover_g'].clip(lower=0.0)
final_df['GDM_g'] = final_df['Dry_Green_g'] + final_df['Dry_Clover_g']
final_df['Dry_Total_g'] = final_df['GDM_g'] + final_df['Dry_Dead_g']

for col in ALL_TARGETS:
    final_df[col] = final_df[col].clip(lower=0.0)

print("\n[6/6] Creating submission...")

submission_rows = []
for _, row in final_df.iterrows():
    image_id = os.path.basename(row['image_path']).replace('.jpg', '')
    for target in cfg.TARGETS:
        submission_rows.append({
            'sample_id': f"{image_id}__{target}",
            'target': row[target]
        })

submission = pd.DataFrame(submission_rows)
submission.to_csv('submission.csv', index=False)

print("\n" + "="*60)
print("COMPLETE!")
print("="*60)
print(f"\nSubmission saved: submission.csv")
print(submission.head(10))
print(f"\nStats:\n{submission['target'].describe()}")

2026-01-28 13:43:03.388266: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1769607783.835847      24 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1769607783.975082      24 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1769607785.035973      24 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1769607785.036023      24 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1769607785.036026      24 computation_placer.cc:177] computation placer alr

Device: cuda
Model: vit_huge_plus_patch16_dinov3.lvd1689m
Models Dir: /kaggle/input/dino-retrain-hu-2/models_trained
CSIRO BIOMASS INFERENCE - VIT_HUGE_PLUS + SIGLIP

[1/6] Loading test data...
Test images: 1

[2/6] Running DINO HUGE inference...
  Loading fold 0...


DINO Inference:   0%|          | 0/1 [00:00<?, ?it/s]

  Loading fold 1...


DINO Inference:   0%|          | 0/1 [00:00<?, ?it/s]

...

Using a slow image processor as `use_fast` is unset and a slow processor was saved with this model. `use_fast=True` will be the default behavior in v4.52, even if the model was saved with a slow processor. This will result in minor differences in outputs. You'll still be able to use a slow processor with `use_fast=False`.


  0%|          | 0/357 [00:00<?, ?it/s]

  Computing test embeddings...
Computing SigLIP embeddings for 1 images...


  0%|          | 0/1 [00:00<?, ?it/s]

  Generating semantic features...

[4/6] Training GBDT models...
  HistGB...
  GB...
  CatBoost...
  LightGBM...

[5/6] Creating ensemble...
Weights: DINO=0.75, SigLIP=0.25

[6/6] Creating submission...

COMPLETE!

Submission saved: submission.csv
                    sample_id     target
0   ID1001187975__Dry_Green_g  32.437280
1    ID1001187975__Dry_Dead_g  33.859824
2  ID1001187975__Dry_Clover_g   0.001323
3         ID1001187975__GDM_g  32.438603
4   ID1001187975__Dry_Total_g  66.298427

Stats:
count     5.000000
mean     33.007091
std      23.447115
min       0.001323
25%      32.437280
50%      32.438603
75%      33.859824
max      66.298427
Name: target, dtype: float64
